# Validation Strategies

Companion notebook for the [Validation Strategies lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/02-validation-strategies).

**The idea in one sentence.** A single train/test split gives a *noisy* estimate of
generalization; cross-validation averages over many splits for a stabler estimate —
but only if you never let information leak from test into train.

What this notebook builds and validates:

- **k-fold CV** and how $k$ trades bias against variance in the estimate.
- **Bootstrap OOB** — a resample leaves out $\approx 1/e \approx 36.8\%$ of points,
  a free held-out set.
- **Data leakage** — preprocessing before the split inflates your score into a lie.
- **Time-series CV** — never train on the future.

We **validate the OOB fraction and that leakage inflates the estimate**, then cover
the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. k-Fold CV Bias-Variance vs k

As k increases, each fold trains on more data (lower bias) but the k estimates become more correlated (higher variance of the CV estimate).

In [ ]:
# Generate a small dataset
n = 200
X = rng.standard_normal((n, 5))
y = (X[:, 0] + 0.5 * X[:, 1] + rng.standard_normal(n) > 0).astype(int)

model = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=300))])

k_values = [2, 3, 5, 10, 20]
means, stds = [], []

for k in k_values:
    scores = cross_val_score(model, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=0))
    means.append(scores.mean())
    stds.append(scores.std())

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(k_values, means, yerr=stds, fmt='o-', color=BRAND, capsize=5,
            linewidth=2, markersize=7, label='Mean CV accuracy ± std')
ax.fill_between(k_values,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.15, color=BRAND)
ax.set_xlabel('k (number of folds)')
ax.set_ylabel('Accuracy')
ax.set_title('k-fold CV: mean accuracy and variance vs k', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for k, m, s in zip(k_values, means, stds):
    print(f'k={k:>2}: mean={m:.4f}  std={s:.4f}')

**What to notice — k trades bias vs variance in the *estimate*.** Small $k$
(e.g. 2) trains on little data, so each fold's model is weak and the CV score is
pessimistically biased; large $k$ (leave-one-out) is nearly unbiased but the folds
overlap heavily, so the estimate is high-variance and expensive. $k=5$–$10$ is the
usual sweet spot.

## 2. Bootstrap OOB Fraction

The fraction of samples *not* selected in a bootstrap sample converges to $1/e \approx 0.368$.

In [ ]:
n_samples = 500
n_bootstrap = 2000
oob_fractions = []

for _ in range(n_bootstrap):
    boot_idx = rng.integers(0, n_samples, n_samples)  # sample with replacement
    oob      = len(set(range(n_samples)) - set(boot_idx)) / n_samples
    oob_fractions.append(oob)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(oob_fractions, bins=40, color=BRAND, alpha=0.7, density=True, edgecolor='none')
ax.axvline(np.mean(oob_fractions), color=TEAL, linewidth=2,
           label=f'Empirical mean = {np.mean(oob_fractions):.4f}')
ax.axvline(1/np.e, color=YELLOW, linewidth=2, linestyle='--',
           label=f'Theoretical 1/e = {1/np.e:.4f}')
ax.set_xlabel('OOB fraction')
ax.set_ylabel('Density')
ax.set_title(f'Bootstrap OOB fraction (n={n_samples}, {n_bootstrap} trials)', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Validate: the bootstrap leaves out ≈ 36.8% of points

Sampling $n$ points with replacement, the chance a given point is *never* drawn is
$(1-1/n)^n \to 1/e \approx 0.368$. That out-of-bag set is a free validation set
(the basis of random-forest OOB error). We check the empirical fraction converges
to $1/e$.

In [ ]:
import numpy as np
rng_v = np.random.default_rng(0)
n = 500
oob_fracs = []
for _ in range(2000):
    drawn = set(rng_v.integers(0, n, n))
    oob_fracs.append((n - len(drawn)) / n)
emp = np.mean(oob_fracs)
print(f'empirical OOB fraction: {emp:.4f}')
print(f'theoretical 1/e:        {1/np.e:.4f}')
assert abs(emp - 1/np.e) < 0.01, 'OOB fraction should converge to 1/e'
print('\n✅ each bootstrap resample leaves out ~36.8% of points as a free validation set')

## 3. Data Leakage Demo

Fitting a scaler on the entire dataset (before splitting) inflates CV accuracy.
Wrapping in a `Pipeline` fixes this automatically.

In [ ]:
n_leak = 300
X_leak = rng.standard_normal((n_leak, 10))
# Hard problem: only first 2 features matter, rest is noise
y_leak = (X_leak[:, 0] - X_leak[:, 1] > 0).astype(int)

# LEAKY: fit scaler on all data before CV
scaler_leaky = StandardScaler()
X_leak_scaled = scaler_leaky.fit_transform(X_leak)  # sees ALL data including val!
clf = LogisticRegression(max_iter=300)
scores_leaky = cross_val_score(clf, X_leak_scaled, y_leak, cv=5)

# CORRECT: scaler inside pipeline, fit per fold
pipe_correct = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=300))])
scores_correct = cross_val_score(pipe_correct, X_leak, y_leak, cv=5)

print(f'Leaky   CV accuracy: {scores_leaky.mean():.4f} ± {scores_leaky.std():.4f}')
print(f'Correct CV accuracy: {scores_correct.mean():.4f} ± {scores_correct.std():.4f}')
print()
if scores_leaky.mean() > scores_correct.mean():
    print('Leaky pipeline reports HIGHER accuracy — this is the data leakage effect.')
else:
    print('Similar scores — try with a larger, more complex dataset to see the gap.')

### Validate: leakage inflates the CV score into a lie

The classic bug: fit a scaler / feature selector on the *whole* dataset, then split.
The test folds have already influenced preprocessing, so CV looks great and
production is a disaster. We compare leaky vs correct pipelines on **pure noise**
(true accuracy = 50%): leakage manufactures signal that isn't there.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

rng_l = np.random.default_rng(1)
Xn = rng_l.normal(size=(200, 5000))     # pure noise: 5000 random features
yn = rng_l.integers(0, 2, 200)          # random labels -> true accuracy is 50%

# LEAKY: select the "best" features using ALL the data, THEN cross-validate
Xsel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
leaky = cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=5).mean()

# CORRECT: selection happens INSIDE each fold via a pipeline
correct = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
    Xn, yn, cv=5).mean()

print(f'leaky CV accuracy   (selection before split): {leaky:.2%}  <- looks predictive!')
print(f'correct CV accuracy (selection inside folds):  {correct:.2%}  <- ~chance, the truth')
assert leaky > correct + 0.1, 'leakage should inflate the estimate above the true ~50%'
print('\n✅ leakage manufactured accuracy from pure noise; the pipeline prevents it')

## 4. Time-Series Cross-Validation

Walk-forward validation: each split trains on all past data and tests on the next window.

In [ ]:
T = 100
n_splits = 5
step = T // (n_splits + 1)

fig, ax = plt.subplots(figsize=(11, 4))

for i in range(n_splits):
    train_end  = step * (i + 1)
    val_start  = train_end
    val_end    = train_end + step

    ax.barh(i, train_end, left=0, color=BRAND, alpha=0.7, height=0.6)
    ax.barh(i, step, left=val_start, color=ROSE, alpha=0.8, height=0.6)
    ax.text(train_end / 2, i, 'Train', ha='center', va='center', color='white', fontsize=9)
    ax.text(val_start + step / 2, i, 'Val', ha='center', va='center', color='white', fontsize=9)

train_patch = mpatches.Patch(color=BRAND, alpha=0.7, label='Train (expanding)')
val_patch   = mpatches.Patch(color=ROSE,  alpha=0.8, label='Validation')
ax.legend(handles=[train_patch, val_patch])
ax.set_xlabel('Time step')
ax.set_yticks(range(n_splits))
ax.set_yticklabels([f'Split {i+1}' for i in range(n_splits)])
ax.set_title('Walk-forward (expanding window) time-series CV', color='white')
ax.set_xlim(0, T)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**What to notice — time-series CV never peeks ahead.** Standard k-fold shuffles,
so a fold can train on Tuesday to predict Monday — impossible in production. The
expanding-window scheme always trains on a prefix and tests on the next block, so
the evaluation matches how the model will actually be used.

---
## ✏️ Your turn

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **preprocessing before split** | scaler/feature-selection leakage inflates CV (demo) — always use a pipeline |
| **k too small / too large** | small = biased-pessimistic, large = high-variance & expensive |
| **shuffling time series** | trains on the future → optimistic garbage; use expanding windows |
| **no stratification** | rare classes can vanish from a fold; use stratified k-fold |
| **tuning on the CV score** | repeated CV-driven tuning overfits the validation folds; keep a final hold-out |

Demo: shuffled k-fold on a time series peeks at the future.

In [ ]:
# k-fold on time series LEAKS: shuffling lets the model see the future. On a trending
# series, shuffled k-fold reports a far better error than the honest expanding-window CV.
rng_t = np.random.default_rng(3)
series_t = np.cumsum(rng_t.normal(0.1, 1, 200))          # a random walk with drift
# shuffled k-fold "score" (predict each point from the global mean) vs expanding window
shuffled_mae = np.mean(np.abs(series_t - series_t.mean()))           # sees all points
exp_maes = []
for cut in range(50, 200, 30):
    exp_maes.append(abs(series_t[cut] - series_t[:cut].mean()))       # only the past
print(f'shuffled-fold MAE (peeks at all data): {shuffled_mae:.2f}')
print(f'expanding-window MAE (past only):      {np.mean(exp_maes):.2f}')
print('\nShuffling time flatters the model; expanding-window CV reflects real deployment.')

### Exercise 1 — `kfold_cv_score(X, y, model, k=5)`

Implement k-fold cross-validation from scratch.

1. Split the dataset into k folds (you can use integer division; drop any remainder)
2. For each fold i: train on all other folds, evaluate on fold i
3. Return an array of k validation accuracy scores

**Hint:** `np.array_split` creates k roughly equal parts.

In [ ]:
def kfold_cv_score(X, y, model, k=5):
    """
    k-fold CV from scratch.
    Returns array of k validation accuracy scores.
    """
    X, y = np.asarray(X), np.asarray(y)
    # TODO(you): split into k folds, train on k-1, evaluate on 1
    # Hint:
    #   idx = np.arange(len(X))
    #   folds = np.array_split(idx, k)
    #   for each fold: train_idx = concat of other folds, val_idx = this fold
    scores = []
    return np.array(scores)

In [ ]:
from sklearn.linear_model import LogisticRegression as LR
from sklearn.model_selection import cross_val_score as sk_cvs

clf_test = LR(max_iter=300, random_state=0)
my_scores = kfold_cv_score(X, y, clf_test, k=5)

assert len(my_scores) == 5, f"Expected 5 scores, got {len(my_scores)}"
assert all(0 <= s <= 1 for s in my_scores), "Scores should be in [0, 1]"

# Compare to sklearn's CV (approx — different fold assignment due to no shuffle)
sk_scores = sk_cvs(LR(max_iter=300), X, y, cv=5)
assert abs(my_scores.mean() - sk_scores.mean()) < 0.10, \
    f"Means differ too much: yours={my_scores.mean():.4f}, sklearn={sk_scores.mean():.4f}"

# Edge case: k = n (leave-one-out) on a small subset — each fold validates on
# exactly 1 sample, so each fold's "accuracy" must be exactly 0.0 or 1.0
X_small, y_small = X[:10], y[:10]
loo_scores = kfold_cv_score(X_small, y_small, LR(max_iter=300, random_state=0), k=10)
assert len(loo_scores) == 10, f"k=n should give n scores, got {len(loo_scores)}"
assert all(s in (0.0, 1.0) for s in loo_scores), \
    "Each leave-one-out fold has 1 validation sample, so its score must be exactly 0.0 or 1.0"

print(f"Your scores:   {my_scores.round(4)}  mean={my_scores.mean():.4f}")
print(f"sklearn scores: {sk_scores.round(4)}  mean={sk_scores.mean():.4f}")
print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def kfold_cv_score(X, y, model, k=5):
    from sklearn.base import clone
    X, y = np.asarray(X), np.asarray(y)
    idx   = np.arange(len(X))
    folds = np.array_split(idx, k)
    scores = []
    for i in range(k):
        val_idx   = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        m = clone(model)
        m.fit(X[train_idx], y[train_idx])
        acc = np.mean(m.predict(X[val_idx]) == y[val_idx])
        scores.append(acc)
    return np.array(scores)
```

</details>

### Exercise 2 — `bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42)`

Simulate bootstrap sampling to estimate the OOB fraction.

For each of `n_bootstrap` trials:
1. Draw `n` samples with replacement from `{0, 1, ..., n-1}`
2. Count how many original indices were NOT selected

Return the average OOB fraction across all trials.

In [ ]:
def bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42):
    """
    Estimate the fraction of samples not included in a bootstrap sample.
    Returns scalar ~ 1/e ≈ 0.368.
    """
    rng_b = np.random.default_rng(seed)
    # TODO(you): for each bootstrap trial, draw n samples with replacement,
    # compute the OOB fraction, and return the mean
    fractions = []
    return np.mean(fractions)

In [ ]:
oob = bootstrap_oob_fraction(n=500, n_bootstrap=5000, seed=42)
assert oob is not None, "returned None"
assert abs(oob - 1/np.e) < 0.01, \
    f"Expected OOB fraction ~{1/np.e:.4f}, got {oob:.4f}"

# Edge case: n=1 — the single sample is always drawn (with replacement, from
# itself), so it is never "out of bag" and the OOB fraction must be exactly 0
oob_n1 = bootstrap_oob_fraction(n=1, n_bootstrap=200, seed=1)
assert abs(oob_n1 - 0.0) < 1e-9, f"With n=1, OOB fraction should be exactly 0.0, got {oob_n1}"

print(f"OOB fraction: {oob:.4f}  (theoretical 1/e = {1/np.e:.4f})")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bootstrap_oob_fraction(n, n_bootstrap=10000, seed=42):
    rng_b = np.random.default_rng(seed)
    fractions = []
    for _ in range(n_bootstrap):
        boot = rng_b.integers(0, n, n)
        oob  = len(set(range(n)) - set(boot)) / n
        fractions.append(oob)
    return float(np.mean(fractions))
```

</details>

### Exercise 3 — `k_fold_cross_validation(X, y, k=5, shuffle=True)` (DML 18)

Open-Deep-ML `18_implement-k-fold-cross-validation` asks for the
**index-splitting** step itself — not a full CV loop — matching the
convention used by scikit-learn's `KFold`: return one
`(train_indices, test_indices)` tuple per fold, as plain Python lists.

- Split `np.arange(len(X))` into `k` folds with `np.array_split` — after
  shuffling first if `shuffle=True`.
- Fold `i`'s indices become the test set; every other fold's indices,
  concatenated (in fold order), become the train set.
- When `shuffle=True`, permute the indices with `np.random.permutation`
  *before* splitting — call `np.random.seed(...)` beforehand for
  reproducibility, exactly like DML's own tests do.

In [ ]:
def k_fold_cross_validation(X, y, k=5, shuffle=True):
    """
    DML 18: return a list of (train_indices, test_indices) tuples, one per fold.
    Both index lists are plain Python ints (not numpy scalars).
    """
    n = len(X)
    # TODO(you): build `indices` (shuffled with np.random.permutation if
    # shuffle=True, else np.arange(n)), split into k folds with
    # np.array_split, then assemble (train, test) index-list pairs
    indices = ...
    folds = ...
    result = []
    return result

In [ ]:
# DML's own tests (reproduced with the same np.random.seed(42) convention)
np.random.seed(42)
folds_a = k_fold_cross_validation(np.arange(10), np.arange(10), k=5, shuffle=False)
assert folds_a == [
    ([2, 3, 4, 5, 6, 7, 8, 9], [0, 1]),
    ([0, 1, 4, 5, 6, 7, 8, 9], [2, 3]),
    ([0, 1, 2, 3, 6, 7, 8, 9], [4, 5]),
    ([0, 1, 2, 3, 4, 5, 8, 9], [6, 7]),
    ([0, 1, 2, 3, 4, 5, 6, 7], [8, 9]),
], f"k=5, shuffle=False mismatch: {folds_a}"

np.random.seed(42)
folds_b = k_fold_cross_validation(np.arange(10), np.arange(10), k=2, shuffle=True)
assert folds_b == [
    ([2, 9, 4, 3, 6], [8, 1, 5, 0, 7]),
    ([8, 1, 5, 0, 7], [2, 9, 4, 3, 6]),
], f"k=2, shuffle=True mismatch: {folds_b}"

np.random.seed(42)
folds_c = k_fold_cross_validation(np.arange(15), np.arange(15), k=3, shuffle=False)
assert folds_c == [
    ([5, 6, 7, 8, 9, 10, 11, 12, 13, 14], [0, 1, 2, 3, 4]),
    ([0, 1, 2, 3, 4, 10, 11, 12, 13, 14], [5, 6, 7, 8, 9]),
    ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14]),
], f"k=3, shuffle=False mismatch: {folds_c}"

# Edge case: k = n — every fold has exactly 1 test index (leave-one-out)
folds_loo = k_fold_cross_validation(np.arange(6), np.arange(6), k=6, shuffle=False)
assert len(folds_loo) == 6, f"k=n should give n folds, got {len(folds_loo)}"
assert all(len(test) == 1 for _, test in folds_loo), "Each leave-one-out fold must test on exactly 1 index"
assert sorted(test[0] for _, test in folds_loo) == list(range(6)), \
    "LOO test indices should cover every sample exactly once"

print("\u2705 Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def k_fold_cross_validation(X, y, k=5, shuffle=True):
    n = len(X)
    indices = np.arange(n)
    if shuffle:
        indices = np.random.permutation(n)
    folds = np.array_split(indices, k)
    result = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        result.append((train_idx.tolist(), test_idx.tolist()))
    return result
```

</details>

## Key takeaways

- **CV averages over splits** for a stabler generalization estimate than one split;
  $k$ trades bias (small $k$) against variance/cost (large $k$) — 5–10 is standard.
- **Bootstrap OOB ≈ 36.8%** ($1/e$) of points are left out per resample — a free
  validation set (we verified it).
- **Leakage is the silent killer:** any preprocessing fit on the full data before
  splitting inflates the score — we manufactured accuracy from pure noise, then
  fixed it with a pipeline.
- **Time-series needs expanding-window CV** — shuffled k-fold trains on the future
  and lies about deployment performance.